# Session 3 — Supervised Fine-Tuning & LoRA**Duration:** 75 min  **Repository:** [MiniMind-Colab](https://github.com/your-org/minimind-colab)| Part | Topic | Time ||------|-------|------|| A | SFT Data & Label Masking | 20 min || B | Full SFT Training | 15 min || C | LoRA — Parameter-Efficient Fine-Tuning | 35 min || — | Wrap-up & Q\&A | 5 min |

In [ ]:
# Cell 2 — Environment setup!pip install torch transformers datasets --quietimport math, json, os, timeimport torchimport torch.nn as nnimport torch.nn.functional as Ffrom torch import optimfrom torch.utils.data import Dataset, DataLoaderfrom contextlib import nullcontextfrom transformers.activations import ACT2FNfrom transformers import PreTrainedModel, GenerationMixin, PretrainedConfig, AutoTokenizerfrom transformers.modeling_outputs import MoeCausalLMOutputWithPastdevice = 'cuda' if torch.cuda.is_available() else 'cpu'print(f'PyTorch {torch.__version__}  CUDA available: {torch.cuda.is_available()}')

## Part A — SFT Data & Label Masking (20 min)In pretraining we predicted *every* token in the sequence.For SFT (Supervised Fine-Tuning) we only want the model to learn the **assistant’s replies**.System prompts and user messages are *context* — not prediction targets.Key idea: **label masking** sets labels to `-100` for tokens we do *not* want to back-propagate through.PyTorch’s `CrossEntropyLoss` ignores positions where `label == -100`.### How it works1. The tokenizer’s `apply_chat_template` produces a string with role markers.2. We tokenize and scan for the **BOS-assistant** marker (`<bos>assistant\n`).3. Everything between BOS-assistant and the next **EOS** marker is an assistant turn — those tokens get real labels.4. All other tokens (system, user) are masked with `-100`.

### Prerequisites from Sessions 1 & 2We re-define the model architecture from previous sessions.These are provided — no changes needed.

In [ ]:
# Cell — Prerequisites: Model architecture (from Sessions 1 & 2)class RMSNorm(torch.nn.Module):    def __init__(self, dim, eps=1e-5):        super().__init__()        self.eps = eps        self.weight = nn.Parameter(torch.ones(dim))    def norm(self, x):        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)    def forward(self, x):        return (self.weight * self.norm(x.float())).type_as(x)def precompute_freqs_cis(dim, end=32*1024, rope_base=1e6):    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[:(dim // 2)].float() / dim))    t = torch.arange(end)    freqs = torch.outer(t, freqs).float()    freqs_cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1)    freqs_sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)    return freqs_cos, freqs_sindef apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):    def rotate_half(x):        return torch.cat((-x[..., x.shape[-1] // 2:], x[..., :x.shape[-1] // 2]), dim=-1)    q_embed = ((q * cos.unsqueeze(unsqueeze_dim)) + (rotate_half(q) * sin.unsqueeze(unsqueeze_dim))).to(q.dtype)    k_embed = ((k * cos.unsqueeze(unsqueeze_dim)) + (rotate_half(k) * sin.unsqueeze(unsqueeze_dim))).to(k.dtype)    return q_embed, k_embeddef repeat_kv(x, n_rep):    bs, slen, num_kv_heads, head_dim = x.shape    if n_rep == 1:        return x    return (x[:, :, :, None, :]            .expand(bs, slen, num_kv_heads, n_rep, head_dim)            .reshape(bs, slen, num_kv_heads * n_rep, head_dim))class MiniMindConfig(PretrainedConfig):    model_type = 'minimind'    def __init__(self, hidden_size=768, num_hidden_layers=8, use_moe=False, **kwargs):        super().__init__(**kwargs)        self.hidden_size = hidden_size        self.num_hidden_layers = num_hidden_layers        self.use_moe = use_moe        self.dropout = kwargs.get('dropout', 0.0)        self.vocab_size = kwargs.get('vocab_size', 6400)        self.bos_token_id = kwargs.get('bos_token_id', 1)        self.eos_token_id = kwargs.get('eos_token_id', 2)        self.flash_attn = kwargs.get('flash_attn', True)        self.num_attention_heads = kwargs.get('num_attention_heads', 8)        self.num_key_value_heads = kwargs.get('num_key_value_heads', 4)        self.head_dim = kwargs.get('head_dim', self.hidden_size // self.num_attention_heads)        self.hidden_act = kwargs.get('hidden_act', 'silu')        self.intermediate_size = kwargs.get('intermediate_size', math.ceil(hidden_size * math.pi / 64) * 64)        self.max_position_embeddings = kwargs.get('max_position_embeddings', 32768)        self.rms_norm_eps = kwargs.get('rms_norm_eps', 1e-6)        self.rope_theta = kwargs.get('rope_theta', 1e6)        self.num_experts = kwargs.get('num_experts', 4)        self.num_experts_per_tok = kwargs.get('num_experts_per_tok', 1)        self.moe_intermediate_size = kwargs.get('moe_intermediate_size', self.intermediate_size)        self.norm_topk_prob = kwargs.get('norm_topk_prob', True)        self.router_aux_loss_coef = kwargs.get('router_aux_loss_coef', 5e-4)class Attention(nn.Module):    def __init__(self, config):        super().__init__()        self.num_key_value_heads = config.num_attention_heads if config.num_key_value_heads is None else config.num_key_value_heads        self.n_local_heads = config.num_attention_heads        self.n_local_kv_heads = self.num_key_value_heads        self.n_rep = self.n_local_heads // self.n_local_kv_heads        self.head_dim = config.head_dim        self.q_proj = nn.Linear(config.hidden_size, config.num_attention_heads * self.head_dim, bias=False)        self.k_proj = nn.Linear(config.hidden_size, self.num_key_value_heads * self.head_dim, bias=False)        self.v_proj = nn.Linear(config.hidden_size, self.num_key_value_heads * self.head_dim, bias=False)        self.o_proj = nn.Linear(config.num_attention_heads * self.head_dim, config.hidden_size, bias=False)        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)        self.attn_dropout = nn.Dropout(config.dropout)        self.resid_dropout = nn.Dropout(config.dropout)        self.dropout = config.dropout        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention') and config.flash_attn    def forward(self, x, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):        bsz, seq_len, _ = x.shape        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)        xq = xq.view(bsz, seq_len, self.n_local_heads, self.head_dim)        xk = xk.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)        xv = xv.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)        xq, xk = self.q_norm(xq), self.k_norm(xk)        cos, sin = position_embeddings        xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)        if past_key_value is not None:            xk = torch.cat([past_key_value[0], xk], dim=1)            xv = torch.cat([past_key_value[1], xv], dim=1)        past_kv = (xk, xv) if use_cache else None        xq = xq.transpose(1, 2)        xk = repeat_kv(xk, self.n_rep).transpose(1, 2)        xv = repeat_kv(xv, self.n_rep).transpose(1, 2)        if self.flash and (seq_len > 1) and (past_key_value is None) and (attention_mask is None or torch.all(attention_mask == 1)):            output = F.scaled_dot_product_attention(xq, xk, xv, dropout_p=self.dropout if self.training else 0.0, is_causal=True)        else:            scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)            scores[:, :, :, -seq_len:] += torch.full((seq_len, seq_len), float('-inf'), device=scores.device).triu(1)            if attention_mask is not None:                scores += (1.0 - attention_mask.unsqueeze(1).unsqueeze(2)) * -1e9            output = self.attn_dropout(F.softmax(scores.float(), dim=-1).type_as(xq)) @ xv        output = output.transpose(1, 2).reshape(bsz, seq_len, -1)        output = self.resid_dropout(self.o_proj(output))        return output, past_kvclass FeedForward(nn.Module):    def __init__(self, config, intermediate_size=None):        super().__init__()        intermediate_size = intermediate_size or config.intermediate_size        self.gate_proj = nn.Linear(config.hidden_size, intermediate_size, bias=False)        self.down_proj = nn.Linear(intermediate_size, config.hidden_size, bias=False)        self.up_proj = nn.Linear(config.hidden_size, intermediate_size, bias=False)        self.act_fn = ACT2FN[config.hidden_act]    def forward(self, x):        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))class MiniMindBlock(nn.Module):    def __init__(self, layer_id, config):        super().__init__()        self.self_attn = Attention(config)        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)        self.mlp = FeedForward(config)    def forward(self, hidden_states, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):        residual = hidden_states        hidden_states = self.input_layernorm(hidden_states)        hidden_states, present_key_value = self.self_attn(hidden_states, position_embeddings, past_key_value, use_cache, attention_mask)        hidden_states = residual + hidden_states        residual = hidden_states        hidden_states = self.post_attention_layernorm(hidden_states)        hidden_states = self.mlp(hidden_states)        hidden_states = residual + hidden_states        return hidden_states, present_key_valueclass MiniMindModel(PreTrainedModel):    config_class = MiniMindConfig    def __init__(self, config):        super().__init__(config)        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)        self.dropout = nn.Dropout(config.dropout)        self.layers = nn.ModuleList([MiniMindBlock(i, config) for i in range(config.num_hidden_layers)])        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)        freqs_cos, freqs_sin = precompute_freqs_cis(config.head_dim, config.max_position_embeddings, config.rope_theta)        self.register_buffer('freqs_cos', freqs_cos, persistent=False)        self.register_buffer('freqs_sin', freqs_sin, persistent=False)    def forward(self, input_ids, past_key_values=None, use_cache=False, attention_mask=None, **kwargs):        hidden_states = self.dropout(self.embed_tokens(input_ids))        seq_len = input_ids.shape[1]        past_len = past_key_values[0][0].shape[1] if past_key_values else 0        pos_emb = (self.freqs_cos[past_len:past_len+seq_len], self.freqs_sin[past_len:past_len+seq_len])        new_kvs = []        for i, layer in enumerate(self.layers):            pkv = past_key_values[i] if past_key_values else None            hidden_states, kv = layer(hidden_states, pos_emb, pkv, use_cache, attention_mask)            new_kvs.append(kv)        hidden_states = self.norm(hidden_states)        aux_loss = torch.tensor(0.0, device=hidden_states.device)        return hidden_states, new_kvs if use_cache else None, aux_lossclass MiniMindForCausalLM(PreTrainedModel, GenerationMixin):    config_class = MiniMindConfig    def __init__(self, config=None):        self.config = config or MiniMindConfig()        super().__init__(self.config)        self.model = MiniMindModel(self.config)        self.lm_head = nn.Linear(self.config.hidden_size, self.config.vocab_size, bias=False)        self.model.embed_tokens.weight = self.lm_head.weight    def forward(self, input_ids, labels=None, **kwargs):        hidden_states, past_key_values, aux_loss = self.model(input_ids, **kwargs)        logits = self.lm_head(hidden_states)        loss = None        if labels is not None:            shift_logits = logits[..., :-1, :].contiguous()            shift_labels = labels[..., 1:].contiguous()            loss = F.cross_entropy(shift_logits.view(-1, self.config.vocab_size), shift_labels.view(-1), ignore_index=-100)        return MoeCausalLMOutputWithPast(loss=loss, aux_loss=aux_loss, logits=logits,                                         past_key_values=past_key_values, hidden_states=hidden_states)print('Model architecture loaded ✓')

### 1. SFTDataset — Chat Data & Label MaskingThe `SFTDataset` loads a JSONL file of multi-turn conversations.Each sample is formatted with the tokenizer’s chat template, thena `generate_labels` function scans for `<bos>assistant\n` and `<eos>\n`markers to decide which tokens the model should predict.**Key insight:** only the assistant’s reply tokens contribute to the loss.System/user tokens are masked with `-100` so `CrossEntropyLoss` ignores them.

In [ ]:
# Cell — SFTDataset implementationclass SFTDataset(Dataset):    def __init__(self, jsonl_path, tokenizer, max_length=1024):        super().__init__()        self.tokenizer = tokenizer        self.max_length = max_length        if jsonl_path and os.path.exists(jsonl_path):            from datasets import load_dataset            self.samples = load_dataset('json', data_files=jsonl_path, split='train')        else:            # Synthetic data for notebook demo            self.samples = [                {'conversations': [                    {'role': 'system', 'content': 'You are a helpful assistant.'},                    {'role': 'user', 'content': f'What is {i} + {i}?'},                    {'role': 'assistant', 'content': f'The answer is {i + i}.'},                ]} for i in range(200)            ]        self.bos_id = tokenizer(f'{tokenizer.bos_token}assistant\n', add_special_tokens=False).input_ids        self.eos_id = tokenizer(f'{tokenizer.eos_token}\n', add_special_tokens=False).input_ids    def __len__(self):        return len(self.samples)    def generate_labels(self, input_ids):        # ============================================================        # TODO: Generate label mask for SFT training        #        # Scan through input_ids looking for assistant turn markers.        # Tokens that are part of an assistant reply get their real        # token ID as the label; all other tokens (system, user,        # padding) are masked with -100.        #        # Algorithm:        #   1. Start with labels = [-100] * len(input_ids)        #   2. Walk through input_ids with index i        #   3. When input_ids[i:i+len(bos_id)] matches self.bos_id,        #      the assistant turn starts at i + len(bos_id)        #   4. Scan forward to find self.eos_id (end of reply)        #   5. Copy input_ids[j] → labels[j] for assistant tokens        #      (from start up to and including the eos_id tokens)        #   6. Advance i past the eos_id; otherwise i += 1        #        # Input:  input_ids — list of ints, length max_length        # Output: labels    — list of ints, same length        #        # Mathematical intuition:        #   labels[j] = input_ids[j]  if j is inside assistant turn        #   labels[j] = -100          otherwise (ignored by CE loss)        #        # Hint: use list slicing (input_ids[i:i+n] == self.bos_id)        #   to match marker sequences; use min() to clamp to max_length        # ============================================================        raise NotImplementedError("Fill in the TODO above")    def __getitem__(self, index):        sample = self.samples[index]        conversations = sample['conversations']        prompt = self.tokenizer.apply_chat_template(conversations, tokenize=False, add_generation_prompt=False)        input_ids = self.tokenizer(prompt).input_ids[:self.max_length]        input_ids += [self.tokenizer.pad_token_id] * (self.max_length - len(input_ids))        labels = self.generate_labels(input_ids)        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)print('SFTDataset class defined ✓')

In [ ]:
# Cell — Tokenizer setup for SFTtokenizer = AutoTokenizer.from_pretrained('HuggingFaceTB/cosmo2-tokenizer', trust_remote_code=True)if tokenizer.pad_token is None:    tokenizer.pad_token = tokenizer.eos_tokenMAX_SEQ_LEN = 256BATCH_SIZE = 8sft_ds = SFTDataset(jsonl_path=None, tokenizer=tokenizer, max_length=MAX_SEQ_LEN)sft_loader = DataLoader(sft_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)print(f'SFT dataset size: {len(sft_ds)}')print(f'Batches: {len(sft_loader)}')

In [ ]:
# Cell — ✅ Milestone 1: Label masking verificationinput_ids, labels = sft_ds[0]total_tokens = labels.numel()masked_tokens = (labels == -100).sum().item()real_tokens = total_tokens - masked_tokensmask_ratio = masked_tokens / total_tokensprint(f'Total tokens:       {total_tokens}')print(f'Masked tokens (-100): {masked_tokens}')print(f'Real label tokens:  {real_tokens}')print(f'Mask ratio:         {mask_ratio:.2%}')# System + user tokens should be masked; at least 30% should be -100assert mask_ratio >= 0.30, (    f'Label masking ratio too low: {mask_ratio:.2%}. '    f'Expected >= 30% of tokens to be masked (-100). '    f'Check that system/user tokens are masked.')# Assistant tokens should have real IDs (not -100)assert real_tokens > 0, 'No real labels found — assistant tokens should have real IDs'print(f'\n✅ Milestone 1 passed — Label masking correct')print(f'   {mask_ratio:.1%} of tokens masked (system/user/pad)')print(f'   {real_tokens} tokens with real labels (assistant replies)')

## Part B — Full SFT Training (15 min)Now we fine-tune the entire model on our SFT dataset.The training loop is identical to pretraining — the only difference is thatour labels mask non-assistant tokens with `-100`.**Full SFT** updates *all* model parameters. This is effective but:- Requires storing a full copy of the model per task- Risk of catastrophic forgetting on other tasksWe’ll address these issues with LoRA in Part C.

In [ ]:
# Cell — Learning rate schedule (provided)def get_lr(current_step, total_steps, lr):    return lr * (0.1 + 0.45 * (1 + math.cos(math.pi * current_step / total_steps)))print('get_lr function defined ✓')

In [ ]:
# Cell — SFT training setupconfig = MiniMindConfig(    hidden_size=128,    num_hidden_layers=2,    num_attention_heads=4,    num_key_value_heads=2,    vocab_size=tokenizer.vocab_size,)torch.manual_seed(42)model = MiniMindForCausalLM(config).to(device)total_params = sum(p.numel() for p in model.parameters()) / 1e6print(f'SFT model: {total_params:.2f}M parameters')learning_rate = 5e-4accumulation_steps = 2num_epochs = 1optimizer = optim.AdamW(model.parameters(), lr=learning_rate)dtype_str = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'use_amp = torch.cuda.is_available()dtype = torch.bfloat16 if dtype_str == 'bfloat16' else torch.float16autocast_ctx = nullcontext() if not use_amp else torch.cuda.amp.autocast(dtype=dtype)scaler = torch.cuda.amp.GradScaler(enabled=(dtype_str == 'float16' and use_amp))total_steps = len(sft_loader) * num_epochsprint(f'Total SFT training steps: {total_steps}')print(f'Mixed precision: {dtype_str if use_amp else "disabled (CPU)"}')

In [ ]:
# Cell — SFT training loopmodel.train()losses = []log_interval = 10global_step = 0for epoch in range(num_epochs):    for step, (input_ids, labels) in enumerate(sft_loader):        input_ids = input_ids.to(device)        labels = labels.to(device)        global_step += 1        lr = get_lr(global_step, total_steps, learning_rate)        for pg in optimizer.param_groups:            pg['lr'] = lr        with autocast_ctx:            res = model(input_ids, labels=labels)            loss = (res.loss + res.aux_loss) / accumulation_steps        scaler.scale(loss).backward()        if global_step % accumulation_steps == 0:            scaler.unscale_(optimizer)            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)            scaler.step(optimizer)            scaler.update()            optimizer.zero_grad(set_to_none=True)        current_loss = loss.item() * accumulation_steps        losses.append(current_loss)        if global_step % log_interval == 0 or global_step == 1:            print(f'Step {global_step:>4d}/{total_steps} | Loss: {current_loss:.4f} | LR: {lr:.6f}')if global_step % accumulation_steps != 0:    scaler.unscale_(optimizer)    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)    scaler.step(optimizer)    scaler.update()    optimizer.zero_grad(set_to_none=True)print(f'\nSFT training complete. Final loss: {losses[-1]:.4f}')

In [ ]:
# Cell — ✅ Milestone 2: SFT loss decreasesfirst_losses = losses[:5]last_losses = losses[-5:]first_avg = sum(first_losses) / len(first_losses)last_avg = sum(last_losses) / len(last_losses)print(f'First 5 losses avg: {first_avg:.4f}')print(f'Last 5 losses avg:  {last_avg:.4f}')print(f'Loss decrease:      {first_avg - last_avg:.4f}')assert last_avg < first_avg, (    f'SFT loss did not decrease! First avg: {first_avg:.4f}, Last avg: {last_avg:.4f}')assert len(losses) >= 20, f'Not enough training steps: {len(losses)}'print(f'\n✅ Milestone 2 passed — SFT loss decreased from {first_avg:.4f} to {last_avg:.4f}')

## Part C — LoRA: Parameter-Efficient Fine-Tuning (35 min)**LoRA** (Low-Rank Adaptation) freezes all pretrained weights and addssmall trainable matrices to each linear layer:$$W' = W + B \cdot A$$Where:- $W$ is the frozen pretrained weight $(d_{out} \times d_{in})$- $A$ is a low-rank matrix $(r \times d_{in})$ — Gaussian init- $B$ is a low-rank matrix $(d_{out} \times r)$ — zero init- $r \ll d_{in}, d_{out}$ (typically 8–64)**Benefits:**- Only ~1–5% of parameters are trainable- Checkpoint size is tiny (a few MB vs hundreds of MB)- Multiple tasks = multiple small LoRA adapters on the same base model- No inference latency increase after merging

### 1. LoRA ModuleThe `LoRA` class is a simple two-layer linear network:- `A`: projects from `in_features` down to `rank` (Gaussian init, std=0.02)- `B`: projects from `rank` back to `out_features` (zero init)Zero-initializing `B` ensures the LoRA contribution starts at zero,so the model behaves identically to the pretrained model at the start of training.

In [ ]:
# Cell — LoRA module implementationclass LoRA(nn.Module):    def __init__(self, in_features, out_features, rank):        super().__init__()        # ============================================================        # TODO: Initialize LoRA low-rank matrices A and B        #        # Create two linear layers (no bias) that form a low-rank        # decomposition: the forward pass computes B(A(x)).        #        # Input:  in_features, out_features, rank        # Output: self.rank, self.A, self.B attributes        #        # Mathematical intuition:        #   A: (in_features → rank)   — Gaussian init, std=0.02        #   B: (rank → out_features)  — zero init        #   Zero-init B means LoRA output starts at 0 (no disruption)        #        # Hint: nn.Linear(..., bias=False), .weight.data.normal_(),        #   .weight.data.zero_()        # ============================================================        raise NotImplementedError("Fill in the TODO above")    def forward(self, x):        # ============================================================        # TODO: Implement LoRA forward pass        #        # Pass input through the low-rank decomposition.        #        # Input:  x — shape (..., in_features)        # Output: tensor of shape (..., out_features)        #        # Mathematical intuition:        #   output = B(A(x))  i.e. project down then back up        #        # Hint: self.A, self.B        # ============================================================        raise NotImplementedError("Fill in the TODO above")print('LoRA class defined ✓')

### 2. Applying LoRA to the ModelThe `apply_lora` function walks through all modules in the model andattaches a LoRA adapter to every **square** linear layer (where`in_features == out_features`). This targets the Q, K, V, O projectionsin attention while skipping non-square layers like the FFN projections.The original forward is monkey-patched to add the LoRA output:`new_forward(x) = original_forward(x) + lora(x)`

In [ ]:
# Cell — apply_lora, save_lora, load_loradef apply_lora(model, rank=16):    # ============================================================    # TODO: Attach LoRA adapters to square linear layers    #    # Walk through all named modules in the model. For each    # nn.Linear where in_features == out_features (square),    # create a LoRA adapter and monkey-patch the forward method.    #    # Steps:    #   1. Iterate model.named_modules()    #   2. Check isinstance(module, nn.Linear) and shape[0] == shape[1]    #   3. Create LoRA(in_f, out_f, rank) on model.device    #   4. Store it as module.lora = lora (setattr)    #   5. Save original_forward = module.forward    #   6. Define new forward: original(x) + lora(x)    #      Use default args to capture variables correctly!    #   7. Replace module.forward = forward_with_lora    #    # Input:  model — nn.Module, rank — int    # Output: None (modifies model in-place)    #    # Mathematical intuition:    #   new_output = W @ x + B @ A @ x = (W + BA) @ x    #    # Hint: model.named_modules(), isinstance(), module.weight.shape,    #   .to(model.device), default function arguments for closure    # ============================================================    raise NotImplementedError("Fill in the TODO above")def save_lora(model, path):    raw_model = getattr(model, '_orig_mod', model)    state_dict = {}    for name, module in raw_model.named_modules():        if hasattr(module, 'lora'):            clean_name = name[7:] if name.startswith('module.') else name            lora_state = {f'{clean_name}.lora.{k}': v.cpu().half() for k, v in module.lora.state_dict().items()}            state_dict.update(lora_state)    torch.save(state_dict, path)def load_lora(model, path):    state_dict = torch.load(path, map_location=model.device)    state_dict = {(k[7:] if k.startswith('module.') else k): v for k, v in state_dict.items()}    for name, module in model.named_modules():        if hasattr(module, 'lora'):            lora_state = {k.replace(f'{name}.lora.', ''): v for k, v in state_dict.items() if f'{name}.lora.' in k}            module.lora.load_state_dict(lora_state)print('apply_lora, save_lora, load_lora defined ✓')

### 3. Merging LoRA WeightsAfter training, we can **merge** the LoRA weights back into the base model:$$W' = W + B \cdot A$$This eliminates any inference overhead — the merged model has the samearchitecture as the original, with no extra parameters.

In [ ]:
# Cell — merge_lora implementationdef merge_lora(model, lora_path, save_path):    # ============================================================    # TODO: Merge LoRA weights into the base model    #    # Load LoRA weights, then for each linear layer compute the    # merged weight W' = W + B @ A and save the full state dict.    #    # Steps:    #   1. Call load_lora(model, lora_path) to load adapter weights    #   2. Get raw_model (handle _orig_mod from torch.compile)    #   3. Build state_dict from raw_model, excluding '.lora.' keys    #   4. For each nn.Linear (not inside .lora.):    #      a. Clone its weight to state_dict    #      b. If it has a .lora attribute, add B.weight @ A.weight    #   5. Save merged state_dict with torch.save    #    # Input:  model, lora_path (str), save_path (str)    # Output: None (saves merged weights to save_path)    #    # Mathematical intuition:    #   W' = W + B @ A    (merge low-rank update into full weight)    #   Use .cpu().half() for memory-efficient storage    #    # Hint: getattr(model, '_orig_mod', model), .state_dict(),    #   module.lora.B.weight.data @ module.lora.A.weight.data    # ============================================================    raise NotImplementedError("Fill in the TODO above")print('merge_lora defined ✓')

### 4. LoRA TrainingWith LoRA applied, we:1. **Freeze** all base parameters2. **Unfreeze** only LoRA parameters (A and B matrices)3. Train with the same SFT loopThis dramatically reduces memory and checkpoint size.

In [ ]:
# Cell — LoRA training setup# Start freshtorch.manual_seed(42)lora_model = MiniMindForCausalLM(config).to(device)# Apply LoRA adapterslora_rank = 16apply_lora(lora_model, rank=lora_rank)# Freeze base parameters, unfreeze LoRAfor param in lora_model.parameters():    param.requires_grad = Falselora_params = []for name, module in lora_model.named_modules():    if hasattr(module, 'lora'):        for param in module.lora.parameters():            param.requires_grad = True            lora_params.append(param)total_params = sum(p.numel() for p in lora_model.parameters())trainable_params = sum(p.numel() for p in lora_params)print(f'Total parameters:     {total_params:,}')print(f'Trainable (LoRA):     {trainable_params:,}')print(f'Trainable ratio:      {trainable_params/total_params:.2%}')lora_lr = 1e-4lora_optimizer = optim.AdamW(lora_params, lr=lora_lr)lora_epochs = 1lora_total_steps = len(sft_loader) * lora_epochsprint(f'\nLoRA rank: {lora_rank}')print(f'LoRA training steps: {lora_total_steps}')

In [ ]:
# Cell — LoRA training looplora_model.train()lora_losses = []log_interval = 10global_step = 0for epoch in range(lora_epochs):    for step, (input_ids, labels) in enumerate(sft_loader):        input_ids = input_ids.to(device)        labels = labels.to(device)        global_step += 1        lr = get_lr(global_step, lora_total_steps, lora_lr)        for pg in lora_optimizer.param_groups:            pg['lr'] = lr        with autocast_ctx:            res = lora_model(input_ids, labels=labels)            loss = (res.loss + res.aux_loss) / accumulation_steps        scaler.scale(loss).backward()        if global_step % accumulation_steps == 0:            scaler.unscale_(lora_optimizer)            torch.nn.utils.clip_grad_norm_(lora_params, 1.0)            scaler.step(lora_optimizer)            scaler.update()            lora_optimizer.zero_grad(set_to_none=True)        current_loss = loss.item() * accumulation_steps        lora_losses.append(current_loss)        if global_step % log_interval == 0 or global_step == 1:            print(f'Step {global_step:>4d}/{lora_total_steps} | Loss: {current_loss:.4f} | LR: {lr:.6f}')if global_step % accumulation_steps != 0:    scaler.unscale_(lora_optimizer)    torch.nn.utils.clip_grad_norm_(lora_params, 1.0)    scaler.step(lora_optimizer)    scaler.update()    lora_optimizer.zero_grad(set_to_none=True)print(f'\nLoRA training complete. Final loss: {lora_losses[-1]:.4f}')

In [ ]:
# Cell — Save and inspect LoRA checkpointlora_save_path = 'lora_checkpoint.pth'save_lora(lora_model, lora_save_path)ckpt_size_mb = os.path.getsize(lora_save_path) / (1024 * 1024)print(f'LoRA checkpoint saved: {lora_save_path}')print(f'Checkpoint size: {ckpt_size_mb:.2f} MB')

In [ ]:
# Cell — ✅ Milestone 3: LoRA parameter efficiencyprint(f'Total parameters:     {total_params:,}')print(f'Trainable (LoRA):     {trainable_params:,}')print(f'Trainable ratio:      {trainable_params/total_params:.2%}')print(f'Checkpoint size:      {ckpt_size_mb:.2f} MB')# LoRA params should be roughly 1-5% of totallora_ratio = trainable_params / total_paramsassert lora_ratio <= 0.05, (    f'LoRA parameter ratio too high: {lora_ratio:.2%}. '    f'Expected <= 5% of total parameters.')# Checkpoint should be smallassert ckpt_size_mb < 10, (    f'LoRA checkpoint too large: {ckpt_size_mb:.2f} MB. '    f'Expected < 10 MB.')print(f'\n✅ Milestone 3 passed — LoRA is parameter-efficient')print(f'   {lora_ratio:.2%} of parameters trainable')print(f'   Checkpoint: {ckpt_size_mb:.2f} MB (< 10 MB)')

In [ ]:
# Cell — Test merge_loramerged_path = 'merged_model.pth'merge_lora(lora_model, lora_save_path, merged_path)merged_size_mb = os.path.getsize(merged_path) / (1024 * 1024)print(f'Merged model saved: {merged_path}')print(f'Merged size: {merged_size_mb:.2f} MB')print(f'LoRA checkpoint was only {ckpt_size_mb:.2f} MB')# Cleanupos.remove(lora_save_path)os.remove(merged_path)print('Temporary files cleaned up ✓')

## Wrap-up (5 min)### What we built today| Component | Key idea ||-----------|----------|| **SFTDataset** | Label masking: only assistant tokens contribute to loss || **Full SFT** | Fine-tune all parameters on chat data || **LoRA** | Low-rank adapters: `W' = W + BA`, ~1–5% trainable params || **apply_lora** | Monkey-patch square linear layers with LoRA forward || **merge_lora** | Fold LoRA weights back: `W' = W + B@A` (zero overhead) |### Key takeaways- SFT teaches the model to follow instructions by masking non-assistant tokens- LoRA achieves comparable quality to full fine-tuning with a fraction of the parameters- LoRA checkpoints are tiny and can be merged for zero-cost inference### References- [LoRA paper (Hu et al. 2021)](https://arxiv.org/abs/2106.09685)- [QLoRA (Dettmers et al. 2023)](https://arxiv.org/abs/2305.14314)- [Chat templates (Hugging Face)](https://huggingface.co/docs/transformers/chat_templating)